# Gemmaverse Deep Dive: Gemma with Ollama, ADK, RAG, and Guardrails

This notebook walks through a practical Gemma workflow: running Gemma-family models with Ollama, connecting them to Google's Agent Development Kit (ADK), adding retrieval with EmbeddingGemma, and applying ShieldGemma-style safety guardrails.

The goal is not only to call a model. By the end, you will have built a small agent stack with four pieces:

1. A local Ollama runtime for serving Gemma models.
2. An ADK tool-using agent that can call Python functions.
3. A retrieval workflow that embeds documents, searches them with FAISS, and grounds answers in retrieved context.
4. A guardrail layer that screens unsafe input before the request reaches the main agent.

## Gemma Model Family

Gemma is Google's family of lightweight open models, built from the same research foundation as Gemini and designed for practical deployment across local, cloud, and experimental environments. The family includes general-purpose language models as well as specialized models for retrieval, safety, vision-language tasks, and other focused use cases.

For this workshop, we use the model names as they appear in the Ollama examples below. If your Ollama library exposes a different Gemma tag, update the model name consistently in the pull commands and in the `LiteLlm` configuration.

## Variants Used in This Notebook

- **Gemma 4 model**: Used as the main reasoning and generation model for the ADK agents.
- **EmbeddingGemma**: Used to convert text into vectors for semantic search and retrieval-augmented generation (RAG).
- **ShieldGemma**: Used as a safety classifier for guardrail examples.

Other Gemma-family models can be substituted as long as they support the same task type. For example, a larger language model may improve reasoning quality, while a smaller one may be better for latency-constrained demos.

## How to Read the Notebook

Run the sections in order. The Ollama setup section is mainly for Google Colab. If you are running this notebook on your own machine and already have Ollama installed and running, you can skip the Colab installation cells and continue from the model pull and API checks.

In [1]:
# @title Install Python requirements when running in Google Colab

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install google-adk -q
    !pip install litellm -q
    !pip install langchain -q
    !pip install langchain-community -q
    !pip install langchain-ollama -q
    !pip install faiss-cpu -q
    print("Colab Python requirements installed.")
else:
    print("Not running in Google Colab. Skipping notebook-level pip installs.")
    print("Use your local environment or project dependency manager instead.")

Colab Python requirements installed.


# 1. Running Gemma with Ollama

Ollama provides a simple local API for downloading, serving, and calling language models. In this notebook, Ollama acts as the model runtime: ADK and LangChain call Ollama over `localhost`, while Ollama handles model loading and inference.

This setup is useful for workshops because it keeps the architecture visible:

1. Pull a model into the local Ollama runtime.
2. Start the Ollama server.
3. Test the model through the command line and HTTP API.
4. Reuse the same runtime from ADK and retrieval components.

You can browse available model tags in the [Ollama model library](https://ollama.com/library). If a model tag in this notebook is not available in your environment, replace it with the appropriate Gemma tag before running the later cells.

## Step 0: Install Ollama in Google Colab Only

Ollama installation is only needed when this notebook is running in Google Colab or another temporary environment that does not already include Ollama.

If you are running locally and already have Ollama installed, skip this cell. Start Ollama from your terminal with `ollama serve`, then continue with the version check and model pull steps.

For Colab, switch the runtime to a GPU first: **Runtime > Change runtime type > GPU**. A T4 GPU or better is recommended for a smoother workshop experience. The cell below installs system packages that help Ollama detect the GPU and then runs the official Ollama installer.

In [2]:
# @title Install Ollama when running in Google Colab

if IN_COLAB:
    !sudo apt update
    !sudo apt install -y pciutils zstd
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print("Not running in Google Colab. Skipping Ollama installation.")
    print("If needed, install Ollama locally from https://ollama.com/download")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
87 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as re

In [ ]:
!ollama list

## Step 1: Start the Ollama Service

Ollama must be running before the notebook can pull models or send inference requests. In Colab, we start it as a background process so the notebook can continue executing later cells.

If you are running locally, you can usually start Ollama outside the notebook with:

```bash
ollama serve
```

The notebook waits a few seconds after launching the service so the API is ready before the next cell checks the version.

In [17]:
# @title Start Ollama service in Colab

import subprocess
import threading
import time

if IN_COLAB:
    def run_ollama_serve():
        subprocess.Popen(["/usr/local/bin/ollama", "serve"])

    thread = threading.Thread(target=run_ollama_serve)
    thread.start()
    time.sleep(5)
    print("Ollama service started in the background.")
else:
    print("Not running in Google Colab. Make sure `ollama serve` is running locally.")

Ollama service started in the background.


In [4]:
!ollama --version

ollama version is 0.30.9


## Step 2: Pull Gemma Models with Ollama

With the Ollama service running, download the models used in the rest of the notebook. The language model handles generation and agent reasoning, EmbeddingGemma creates vectors for retrieval, and ShieldGemma supports the safety guardrail example.

Model downloads can take several minutes depending on model size, network speed, and runtime resources. Colab runtimes are temporary, so pulled models may need to be downloaded again in a future session.

In [5]:
!ollama pull gemma4
!ollama pull shieldgemma
!ollama pull embeddinggemma
#!ollama pull functiongemma

> **Checkpoint:** After the pull commands finish, `ollama list` should show the downloaded models. The quick command-line and HTTP calls below confirm that the Ollama server is responding before we connect it to ADK.

In [6]:
!ollama list

NAME                     ID              SIZE      MODIFIED               
embeddinggemma:latest    85462619ee72    621 MB    Less than a second ago    
shieldgemma:latest       c82cacd5af5e    5.8 GB    6 seconds ago             
gemma4:latest            c6eb396dbd59    9.6 GB    24 seconds ago            


In [7]:
!ollama run gemma4 "roses are red"

Thinking...
Thinking Process:

1.  **Analyze the input:** The user provided "roses are red". This is the c
classic opening line of a poem/rhyming couplet (often associated with nurse
nursery rhymes, lighthearted verses, or simple poetic forms).
2.  **Determine the intent:** When a user provides an incomplete poetic sta
start, they are almost certainly looking for:
    *   The completion of the rhyme scheme (AABB or ABCB).
    *   A playful continuation that matches the tone.
3.  **Recall common continuations/themes:** The standard structure is:
    *   Roses are red,
    *   Violets are blue,
    *   Sugar is sweet,
    *   And so are you. (The most famous completion.)

4.  **Formulate the response:** Provide the commonly accepted and expected 
punchline/continuation to satisfy the user's implied prompt.

5.  **Draft the output:** Complete the traditional rhyme.
...done thinking.

Roses are red,
Violets are blue,
Sugar is sweet,
And so are you.



In [8]:
!curl http://localhost:11434/api/generate -d '{\
      "model": "gemma4",\
      "prompt":"roses are red"\
}'

{"model":"gemma4","created_at":"2026-06-18T00:22:17.076516059Z","response":"vio","done":false}
{"model":"gemma4","created_at":"2026-06-18T00:22:17.08258262Z","response":"lets","done":false}
{"model":"gemma4","created_at":"2026-06-18T00:22:17.089623821Z","response":" are","done":false}
{"model":"gemma4","created_at":"2026-06-18T00:22:17.095134311Z","response":" blue","done":false}
{"model":"gemma4","created_at":"2026-06-18T00:22:17.100434062Z","response":",","done":false}
{"model":"gemma4","created_at":"2026-06-18T00:22:17.105722803Z","response":"\n\n","done":false}
{"model":"gemma4","created_at":"2026-06-18T00:22:17.111066393Z","response":"true","done":false}
{"model":"gemma4","created_at":"2026-06-18T00:22:17.116403264Z","response":" enough","done":false}
{"model":"gemma4","created_at":"2026-06-18T00:22:17.121742294Z","response":" to","done":false}
{"model":"gemma4","created_at":"2026-06-18T00:22:17.127116355Z","response":" love","done":false}
{"model":"gemma4","created_at":"2026-06-1

In [9]:
# @title Notebook imports and shared configuration

# Standard library imports
import asyncio
import json
import logging
import urllib.error
import urllib.request
import warnings
from typing import Any

# Third-party imports
import numpy as np
import requests

# ADK imports
from google.adk.agents import Agent, BaseAgent
from google.adk.models import LlmResponse
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import FunctionTool
from google.adk.tools.agent_tool import AgentTool
from google.genai import types

# LangChain imports
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

# Runtime configuration
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.ERROR)

GENERATION_MODEL = "ollama_chat/gemma4:latest"
EMBEDDING_MODEL = "embeddinggemma"
SHIELD_MODEL = "shieldgemma"
OLLAMA_BASE_URL = "http://localhost:11434"

print("Notebook imports and shared configuration loaded.")

/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()
/tmp/ipykernel_3525/3355439598.py:28: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Notebook imports and shared configuration loaded.


# 2. Building a Tool-Using Agent with ADK

Now that Ollama is serving a Gemma model, we can connect it to Google's **Agent Development Kit (ADK)**. This section starts with a deliberately small agent: a weather assistant that must call a Python function instead of inventing weather data.

This example introduces the core ADK pattern used later in the notebook:

1. Define a tool as a normal Python function.
2. Configure a Gemma-backed ADK agent with instructions and tools.
3. Create a session and runner.
4. Send user messages and inspect tool-call events.

The weather data is intentionally static. The point of this section is tool orchestration, not live weather retrieval.

In [10]:
# @title Step 1: Define Agent Tools

def get_weather(city: str) -> dict:
    """Return weather information for a city."""

    city_aliases = {
        "sfo": "san francisco",
        "sf": "san francisco",
        "san francisco": "san francisco",
        "nyc": "new york",
        "la": "los angeles",
    }

    normalized_city = city.strip().lower()
    normalized_city = city_aliases.get(normalized_city, normalized_city)

    weather_data = {
        "san francisco": {
            "status": "success",
            "report": "The weather in San Francisco is 62°F and partly cloudy.",
        },
        "new york": {
            "status": "success",
            "report": "The weather in New York is 75°F and sunny.",
        },
        "los angeles": {
            "status": "success",
            "report": "The weather in Los Angeles is 78°F and clear.",
        },
    }

    if normalized_city in weather_data:
        return weather_data[normalized_city]

    return {
        "status": "error",
        "error_message": f"Sorry, I don't have weather information for '{city}'.",
    }

# Example tool usage (optional test)
print(get_weather("New York"))
print(get_weather("Paris"))



{'status': 'success', 'report': 'The weather in New York is 75°F and sunny.'}
{'status': 'error', 'error_message': "Sorry, I don't have weather information for 'Paris'."}


In [11]:
# @title Step 2: Define the Weather Agent
model = LiteLlm(model=GENERATION_MODEL)


weather_agent = Agent(
    name="weather_agent",
    model=model,
    description=(
        "Provides concise weather information for specific cities using the get_weather tool."
    ),
    instruction=(
        "You are a helpful weather assistant.\n\n"
        "Rules:\n"
        "- Respond directly to the user. Do not explain your reasoning.\n"
        "- Do not describe what you are going to do unless necessary.\n"
        "- If the user greets you, greet them briefly and ask what city they want weather for.\n"
        "- When the user asks for weather, identify the city and call the `get_weather` tool.\n"
        "- Treat common airport/city aliases as cities when obvious. For example, SFO means San Francisco.\n"
        "- If no city is provided, ask the user for the city.\n"
        "- If the tool succeeds, summarize the weather clearly and concisely.\n"
        "- If the tool returns an error, politely say you could not find weather for that city.\n"
        "- Never invent weather information. Only use the tool result."
    ),
    tools=[get_weather],
)

In [12]:
# @title Step 3: Create Helper Functions to Interact with Any ADK Agent
# --- Session Management ---
# Key Concept: SessionService stores conversation history & state.
# InMemorySessionService is simple, non-persistent storage for this tutorial.

def get_session_service() -> InMemorySessionService:
    """Create and return the session service used by the application."""
    return InMemorySessionService()


async def get_or_create_session(
    session_service: InMemorySessionService,
    app_name: str,
    user_id: str,
    session_id: str,
):
    """Retrieve an existing session or create it if it does not exist."""

    session = await session_service.get_session(
        app_name=app_name,
        user_id=user_id,
        session_id=session_id,
    )

    if session is None:
        session = await session_service.create_session(
            app_name=app_name,
            user_id=user_id,
            session_id=session_id,
        )
        print(
            f"Session created: "
            f"App='{app_name}', User='{user_id}', Session='{session_id}'"
        )
    else:
        print(
            f"Using existing session: "
            f"App='{app_name}', User='{user_id}', Session='{session_id}'"
        )

    return session


def print_tool_events(event) -> None:
    """Print tool calls and tool responses from an ADK event."""

    if not event.content or not event.content.parts:
        return

    for part in event.content.parts:
        function_call = getattr(part, "function_call", None)
        function_response = getattr(part, "function_response", None)

        if function_call:
            print("\n🔧 Tool called")
            print(f"Name: {function_call.name}")
            print(f"Args: {function_call.args}")

        if function_response:
            print("\n✅ Tool response")
            print(f"Name: {function_response.name}")
            print(f"Response: {function_response.response}")


async def call_agent_async(
    agent: BaseAgent,
    query: str,
    app_name: str,
    user_id: str,
    session_id: str,
    session_service: InMemorySessionService,
    debug: bool = True,
) -> str:
    """Run an ADK agent and optionally print tool-call events."""

    print(f"\n>>> User Query: {query}")

    content = types.Content(
        role="user",
        parts=[
            types.Part(text=query),
        ],
    )

    await get_or_create_session(
        session_service=session_service,
        app_name=app_name,
        user_id=user_id,
        session_id=session_id,
    )

    runner = Runner(
        agent=agent,
        app_name=app_name,
        session_service=session_service,
    )

    final_response_text = "Agent did not produce a final response."
    captured_final_response = False

    async for event in runner.run_async(
        user_id=user_id,
        session_id=session_id,
        new_message=content,
    ):
        if debug:
            print_tool_events(event)

        if event.is_final_response() and not captured_final_response:
            captured_final_response = True
            if event.content and event.content.parts:
                final_response_text = event.content.parts[0].text or ""
            elif event.actions and event.actions.escalate:
                final_response_text = (
                    f"Agent escalated: "
                    f"{event.error_message or 'No specific message.'}"
                )
            # Keep draining the ADK stream so tracing cleanup runs in-context.

    return final_response_text

In [13]:
# @title Step 4: Start an Interactive Conversation with the Agent

APP_NAME = "weather_app"
USER_ID = "test_user"
SESSION_ID = "test_session"

session_service = get_session_service()

while True:
    query = input("\nYou: ")
    if query.lower() in {"exit", "quit", "bye"}:
        print("Ending conversation...")
        break

    response = await call_agent_async(
        agent=weather_agent,
        query=query,
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID,
        session_service=session_service,
    )

    print(f"\nAgent: {response}")



You: what is the weather for plano ,tx

>>> User Query: what is the weather for plano ,tx
Session created: App='weather_app', User='test_user', Session='test_session'

🔧 Tool called
Name: get_weather
Args: {'city': 'Plano'}

✅ Tool response
Name: get_weather
Response: {'status': 'error', 'error_message': "Sorry, I don't have weather information for 'Plano'."}

Agent: I could not find weather for that city.

You: give me weather for the city that you know

>>> User Query: give me weather for the city that you know
Using existing session: App='weather_app', User='test_user', Session='test_session'

Agent: 1.  **Analyze the user's request:** The user is asking for "weather for the city that you know."
2.  **Recall constraints/rules:**
    *   If no city is provided, ask the user for the city. (This applies here.)
    *   Do not explain reasoning or describe actions unless necessary.
3.  **Determine the action:** Since the user hasn't specified a new city and seems to be prompting the ass

ERROR:LiteLLM:LoggingWorker error: 
Traceback (most recent call last):
  File "/usr/lib/python3.12/asyncio/tasks.py", line 520, in wait_for
    return await fut
           ^^^^^^^^^
asyncio.exceptions.CancelledError

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/litellm/litellm_core_utils/logging_worker.py", line 99, in _process_log_task
    await asyncio.wait_for(
  File "/usr/lib/python3.12/asyncio/tasks.py", line 519, in wait_for
    async with timeouts.timeout(timeout):
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/timeouts.py", line 115, in __aexit__
    raise TimeoutError from exc_val
TimeoutError



🔧 Tool called
Name: get_weather
Args: {'city': 'New York'}

✅ Tool response
Name: get_weather
Response: {'status': 'success', 'report': 'The weather in New York is 75°F and sunny.'}

Agent: The weather in New York is 75°F and sunny.

You: exit
Ending conversation...


# 3. Building a Retrieval Agent with EmbeddingGemma and Ollama

The previous section showed an agent calling a structured tool. This section adds retrieval so the agent can answer from an external knowledge base instead of relying only on the language model's parameters.

The retrieval flow has four steps:

1. **Embed documents** with EmbeddingGemma through Ollama.
2. **Index vectors** in FAISS for efficient similarity search.
3. **Retrieve context** for each user question.
4. **Generate grounded answers** with the Gemma language model and the retrieved text.

We first build a small embedding client manually so the API contract is clear. Then we switch to LangChain's `OllamaEmbeddings` wrapper to reduce boilerplate for the FAISS-backed knowledge base.

In [14]:
# @title Step 1: Build Embeddings Component with Ollama

class RawOllamaEmbeddingsClient:
    """Generate normalized embeddings using Ollama directly."""

    def __init__(
        self,
        model_name: str = EMBEDDING_MODEL,
        base_url: str = OLLAMA_BASE_URL,
        timeout: float = 60.0,
        max_concurrency: int = 10,
    ) -> None:
        self.model_name = model_name
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.max_concurrency = max_concurrency

    async def embed(self, text: str) -> np.ndarray:
        """Generate one normalized embedding vector."""
        response = await asyncio.to_thread(
            self._embed_sync,
            text,
        )

        embedding = self._extract_embedding(response)
        vector = np.array(embedding, dtype=np.float32)

        return self._normalize(vector)

    async def embed_batch(
        self,
        texts: list[str],
    ) -> list[np.ndarray]:
        """Generate embeddings concurrently with a max concurrency limit."""
        semaphore = asyncio.Semaphore(self.max_concurrency)

        async def _worker(text: str) -> np.ndarray:
            async with semaphore:
                return await self.embed(text)

        return await asyncio.gather(
            *(_worker(text) for text in texts)
        )

    def _embed_sync(self, text: str) -> dict[str, Any]:
        """Call Ollama embedding endpoint synchronously."""
        payload = json.dumps(
            {
                "model": self.model_name,
                "input": text,
            }
        ).encode("utf-8")

        request = urllib.request.Request(
            f"{self.base_url}/api/embed",
            data=payload,
            headers={"Content-Type": "application/json"},
            method="POST",
        )

        try:
            with urllib.request.urlopen(
                request,
                timeout=self.timeout,
            ) as response:
                return json.loads(
                    response.read().decode("utf-8")
                )

        except urllib.error.URLError as exc:
            raise RuntimeError(
                f"Failed to call Ollama embeddings API "
                f"at {self.base_url}: {exc}"
            ) from exc

    @staticmethod
    def _extract_embedding(
        response: dict[str, Any],
    ) -> list[float]:
        """Extract one embedding vector from Ollama response."""
        embeddings = response.get("embeddings")

        if (
            isinstance(embeddings, list)
            and embeddings
            and isinstance(embeddings[0], list)
        ):
            return embeddings[0]

        embedding = response.get("embedding")

        if isinstance(embedding, list):
            return embedding

        raise RuntimeError(
            "Ollama embeddings response did not include an embedding."
        )

    @staticmethod
    def _normalize(vector: np.ndarray) -> np.ndarray:
        """Normalize an embedding vector for cosine similarity search."""
        norm = np.linalg.norm(vector)

        if norm > 0:
            return vector / norm

        return vector

In [18]:
# @title Step 2: Test the Embeddings Component

raw_embeddings = RawOllamaEmbeddingsClient(EMBEDDING_MODEL)

vector = await raw_embeddings.embed(
    "Gemma is a family of lightweight open models."
)

vectors = await raw_embeddings.embed_batch([
        "Hello world",
        "How are you?",
        "EmbeddingGemma is optimized for retrieval.",
])

print(f"Vector: {vector}")
print(f"Vectors: {vectors}")

Vector: [-9.75381881e-02 -6.41951859e-02 -1.85917970e-02 -5.29957265e-02
 -3.17249587e-03  5.52930273e-02 -2.40767840e-02  5.51707633e-02
  1.41664902e-02 -6.81692688e-03 -1.56621356e-02  6.59879297e-03
  2.85615167e-03 -5.78741282e-02 -2.51978822e-02  1.45752504e-02
  7.93488137e-03  1.30586671e-02  3.53757776e-02 -2.53496710e-02
  6.31393194e-02 -3.33737954e-02  1.07520502e-02 -2.65771579e-02
  4.84959893e-02  1.16463033e-02  2.33211089e-02 -1.56723149e-02
  3.09504960e-02  1.55025674e-03 -1.83949305e-03  1.52161764e-02
  5.40940277e-02  5.69015145e-02 -1.19959330e-02  8.84699542e-03
 -1.72920823e-02 -9.37225949e-03 -1.10612707e-02 -2.62972172e-02
  4.66552489e-02  6.29364625e-02 -1.81918312e-02 -1.65953655e-02
 -7.47061670e-02  6.34121057e-03 -5.65144196e-02  2.17175228e-03
  2.31976360e-02  2.72226147e-02 -1.35915410e-02 -5.62172644e-02
 -4.34911922e-02  1.95743721e-02 -2.37847306e-02  2.70304363e-02
  4.15210333e-03 -5.40261576e-03 -8.13885182e-02  3.24972011e-02
 -6.12649843e-02 

In [19]:
# @title Step 3: Simplify Embeddings with LangChain
embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=OLLAMA_BASE_URL,
)

vector = embeddings.embed_query(
    "What is EmbeddingGemma used for?"
)

print(f"Dimension: {len(vector)}")
print(vector[:10])

Dimension: 768
[-0.12100658, -0.06238405, 0.0014681403, 0.043507855, 0.038663015, 0.06061147, 0.0028051022, 0.05095466, 0.03370106, 0.0068001156]


In [20]:
# @title Step 4: Build KnowledgeBase component with FAISS and LangChain

class KnowledgeBase:
    """Knowledge base backed by a LangChain FAISS vector store."""

    def __init__(
        self,
        embeddings,
        documents: list[Document] | None = None,
    ) -> None:
        self.embeddings = embeddings
        self.vector_store: FAISS | None = None

        if documents:
            self.build(documents)

    def build(
        self,
        documents: list[Document],
    ) -> None:
        """Create the FAISS vector store from documents."""

        self.vector_store = FAISS.from_documents(
            documents=documents,
            embedding=self.embeddings,
        )

    def search(
        self,
        query: str,
        k: int = 3,
    ) -> list[Document]:
        """Return only the most relevant documents."""

        if self.vector_store is None:
            return []

        return self.vector_store.similarity_search(
            query,
            k=k,
        )

    def search_with_scores(
        self,
        query: str,
        k: int = 3,
    ) -> list[tuple[Document, float]]:
        """Return relevant documents with relevance scores."""

        if self.vector_store is None:
            return []

        return self.vector_store.similarity_search_with_relevance_scores(
            query,
            k=k,
        )

    def as_retriever(
        self,
        k: int = 3,
    ):
        """Expose the knowledge base as a LangChain retriever."""

        if self.vector_store is None:
            raise ValueError("Knowledge base has not been built yet.")

        return self.vector_store.as_retriever(
            search_kwargs={"k": k}
        )

    def format_search_results(
        self,
        results: list[tuple[Document, float]],
    ) -> str:
        """Format retrieved documents and scores as readable text."""

        if not results:
            return "No relevant documents were found."

        blocks = []

        for rank, (doc, score) in enumerate(results, start=1):
            blocks.append(
                f"""
                Document {rank}
                Relevance: {score:.4f}
                Content:
                {doc.page_content}
                """.strip()
            )

        return "\n\n" + "-" * 80 + "\n\n".join(blocks)

In [21]:
# @title Step 5: Test Knowledge Base

documents = [
    Document(page_content="Gemma is a family of lightweight open models."),
    Document(page_content="EmbeddingGemma is useful for semantic search."),
    Document(page_content="RAG combines retrieval with generation."),
]

embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=OLLAMA_BASE_URL,
)

knowledge_base = KnowledgeBase(
    embeddings=embeddings,
    documents=documents,
)

results = knowledge_base.search_with_scores(
    "What is EmbeddingGemma used for?",
    k=3,
)

print(
    knowledge_base.format_search_results(results)
)



--------------------------------------------------------------------------------Document 1
                Relevance: 0.6346
                Content:
                EmbeddingGemma is useful for semantic search.

Document 2
                Relevance: 0.1666
                Content:
                Gemma is a family of lightweight open models.

Document 3
                Relevance: 0.0849
                Content:
                RAG combines retrieval with generation.


In [22]:
# @title Step 6: Build Agent Factory

def create_knowledge_retrieval_agent(
    *,
    name: str,
    description: str,
    reasoning_model: LiteLlm,
    knowledge_base: KnowledgeBase,
    top_k: int = 3,
) -> BaseAgent:
    """Create an ADK agent backed by a knowledge base."""

    async def search_knowledge_base(
        query: str,
    ) -> str:
        """
        Search the knowledge base and return formatted results.
        """
        if not query.strip():
            return "No query was provided."

        results = knowledge_base.search_with_scores(
            query=query,
            k=max(1, top_k),
        )

        if not results:
            return "No relevant documents were found."

        return knowledge_base.format_search_results(
            results
        )

    search_knowledge_base_tool = FunctionTool(
        func=search_knowledge_base,
    )

    return Agent(
        name=name,
        model=reasoning_model,
        description=description,
        instruction=f"""
            You are a specialist knowledge-base agent.

            Specialty:
            {description}

            Use search_knowledge_base whenever the user's request requires your knowledge base.

            Rules:
            - Answer only from retrieved knowledge-base content.
            - Be clear, concise, and direct.
            - Do not mention tool calls or internal reasoning.
            - If the knowledge base does not contain enough information, say so.
        """.strip(),
        tools=[search_knowledge_base_tool],
    )

In [23]:
# @title Step 7: Test Agent Factory

embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL,
    base_url=OLLAMA_BASE_URL,
)

reasoning_model = LiteLlm(
    model=GENERATION_MODEL,
)

gemma_documents = [
    Document(page_content="Gemma is a family of lightweight open models."),
    Document(page_content="EmbeddingGemma is useful for semantic search and retrieval."),
    Document(page_content="ShieldGemma is used for safety classification and guardrails."),
]

gemma_knowledge_base = KnowledgeBase(
    embeddings=embeddings,
    documents=gemma_documents,
)

gemma_kb_agent = create_knowledge_retrieval_agent(
    name="gemma_knowledge_agent",
    description="Answers questions about Gemma, EmbeddingGemma, ShieldGemma, and Gemma-based RAG.",
    reasoning_model=reasoning_model,
    knowledge_base=gemma_knowledge_base,
    top_k=3,
)


In [24]:
# @title Step 8: Run agent

APP_NAME = "test_app"
USER_ID = "test_user"
SESSION_ID = "test_session"

session_service = get_session_service()

response = await call_agent_async(
    agent=gemma_kb_agent,
    query="What is EmbeddingGemma used for?",
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
    session_service=session_service,
)

print(f"\nAgent: {response}")


>>> User Query: What is EmbeddingGemma used for?
Session created: App='test_app', User='test_user', Session='test_session'

🔧 Tool called
Name: search_knowledge_base
Args: {'query': 'What is EmbeddingGemma used for?'}

✅ Tool response
Name: search_knowledge_base
Response: {'result': '\n\n--------------------------------------------------------------------------------Document 1\n                Relevance: 0.6309\n                Content:\n                EmbeddingGemma is useful for semantic search and retrieval.\n\nDocument 2\n                Relevance: 0.3252\n                Content:\n                ShieldGemma is used for safety classification and guardrails.\n\nDocument 3\n                Relevance: 0.1666\n                Content:\n                Gemma is a family of lightweight open models.'}

Agent: The user asked what EmbeddingGemma is used for. The knowledge base returned content from multiple documents, and Document 1 specifically states: "EmbeddingGemma is useful for sema

# 4. Adding Safety Guardrails with ShieldGemma

The final section adds a safety layer in front of the agent. ShieldGemma-style models are designed to classify content against a safety policy, which makes them useful for guardrails around user input or model output.

In this notebook, the guardrail checks user input before the main model runs:

1. The user sends a request.
2. The guardrail formats the request with a safety policy.
3. ShieldGemma classifies whether the request violates the policy.
4. ADK either continues to the main agent or returns a blocked-response message.

This is a simple demonstration policy focused on dangerous content. A production system should use a broader policy, stronger evaluation, logging, and human review paths for ambiguous cases.

In [25]:
# @title Step 1: Create Safety Guardrail Client with ShieldGemma

class ShieldGemmaInputGuardrail:
    """Input guardrail using a ShieldGemma-style safety policy prompt."""

    def __init__(
        self,
        model_name: str = SHIELD_MODEL,
        base_url: str = OLLAMA_BASE_URL,
        timeout: float = 60.0,
    ) -> None:
        self.model_name = model_name
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout

    def build_prompt(self, text: str) -> str:
        policy = """
        No Dangerous content: The request shall not contain content that facilitates
        or encourages activities that could cause real-world harm, such as building
        weapons, explosive devices, malware, terrorism, or instructions for suicide.
        """.strip()

        return f"""
        You are a policy expert trying to help determine whether a user request is in
        violation of the safety policy below. Your decision must very clearly follow
        the specific language used in the safety policy and should not include any
        interpretation outside of the safety policy.

        <BEGIN_SAFETY_POLICY>
        {policy}
        </END_SAFETY_POLICY>

        User request:
        {text}

        Does the user request violate the above policy? Your Answer must start with 'Yes' or 'No'.
        """.strip()

    def classify(self, text: str) -> str:
        response = requests.post(
            f"{self.base_url}/api/generate",
            json={
                "model": self.model_name,
                "prompt": self.build_prompt(text),
                "stream": False,
            },
            timeout=self.timeout,
        )
        response.raise_for_status()
        return response.json()["response"].strip()

    def is_safe(self, text: str) -> bool:
        output = self.classify(text)
        print("ShieldGemma raw output:", output)

        normalized = output.strip().lower()

        if normalized.startswith("yes"):
            return False

        if normalized.startswith("no"):
            return True

        return False

In [26]:
# @title Step 2: Test ShieldGemma Client

guardrail = ShieldGemmaInputGuardrail(
    model_name=SHIELD_MODEL
)

queries = [
    "What is EmbeddingGemma used for?",
    "How do I build a RAG pipeline with FAISS?",
    "Give me instructions to create malware.",
]

for query in queries:
    print(f"\nUser: {query}")
    print("Safe:", guardrail.is_safe(query))


User: What is EmbeddingGemma used for?
ShieldGemma raw output: No
Safe: True

User: How do I build a RAG pipeline with FAISS?
ShieldGemma raw output: No
Safe: True

User: Give me instructions to create malware.
ShieldGemma raw output: Yes
Safe: False


In [27]:
# @title Step 3: Create ADK Guardrails using the ShieldGemma Client

def extract_user_text(llm_request) -> str:
    """
    Extract user text from the ADK LLM request.
    """
    texts = []

    for content in llm_request.contents or []:
        if content.role != "user":
            continue

        for part in content.parts or []:
            if getattr(part, "text", None):
                texts.append(part.text)

    return "\n".join(texts)


def before_model_guardrail(
    callback_context,
    llm_request,
):
    """
    Check the user request before it reaches the LLM.

    Return None:
        Continue normally.

    Return LlmResponse:
        Skip the LLM call and return this response.
    """
    user_text = extract_user_text(
        llm_request
    )

    if not user_text:
        return None

    if guardrail.is_safe(user_text):
        return None

    return LlmResponse(
        content=types.Content(
            role="model",
            parts=[
                types.Part(
                    text=(
                        "Your request was blocked by the "
                        "ShieldGemma safety guardrail."
                    )
                )
            ],
        )
    )


In [28]:
# @title Step 4: Create Ultimate Agent
rag_documents = [
    Document(
        page_content="""RAG combines retrieval and generation to provide grounded answers."""
    ),
    Document(
        page_content="""
        FAISS is a library for efficient vector similarity search.
        """
    ),
    Document(
        page_content="""
        A retriever returns the most relevant documents
        for a user query.
        """
    ),
]

rag_knowledge_base = KnowledgeBase(
    embeddings=embeddings,
    documents=rag_documents,
)

rag_kb_agent = create_knowledge_retrieval_agent(
    name="rag_knowledge_agent",
    description="""
Answers questions about RAG, embeddings, vector databases,
retrievers, and FAISS.
""",
    reasoning_model=reasoning_model,
    knowledge_base=rag_knowledge_base,
)

In [29]:
# @title Step 5: Create a Space Specialist Agent

space_documents = [
    Document(
        page_content="""
Space exploration uses robotic spacecraft, satellites, landers, rovers,
and crewed missions to study planets, moons, asteroids, and deep space.
"""
    ),
    Document(
        page_content="""
Mars rovers use cameras, spectrometers, drills, and autonomous navigation
systems to study the planet's geology and search for signs of past habitability.
"""
    ),
    Document(
        page_content="""
Telescopes such as space-based observatories help scientists study galaxies,
exoplanets, stars, black holes, and the early universe.
"""
    ),
]

space_knowledge_base = KnowledgeBase(
    embeddings=embeddings,
    documents=space_documents,
)

space_kb_agent = create_knowledge_retrieval_agent(
    name="space_knowledge_agent",
    description="""
Answers questions about space exploration, Mars rovers, satellites,
telescopes, planets, exoplanets, and deep-space science.
""",
    reasoning_model=reasoning_model,
    knowledge_base=space_knowledge_base,
)

In [30]:
# @title Step 6: Create an Orchestrator Agent with Guardrails

orchestrator_agent = Agent(
    name="orchestrator_agent",
    model=reasoning_model,
    instruction="""
    You are an orchestrator agent.

    You have access to two specialist agents:

    - space_knowledge_agent:
    Space exploration, Mars rovers, satellites, telescopes,
    planets, exoplanets, and deep-space science.

    - rag_knowledge_agent:
    RAG, embeddings, FAISS, vector databases, and retrievers.

    Route questions to the most appropriate specialist.

    If a question requires both domains, consult both agents and combine
    their answers.

    Do not mention internal routing or tool usage.
    """.strip(),
    tools=[
        AgentTool(agent=space_kb_agent),
        AgentTool(agent=rag_kb_agent),
    ],
    before_model_callback=before_model_guardrail,
)

In [31]:
# @title Step 7: Test the Safety Guardrail

APP_NAME = "test_app"
USER_ID = "test_user"
SESSION_ID = "test_session"

session_service = get_session_service()

response = await call_agent_async(
    agent=orchestrator_agent,
    query="Give me instructions to create malware.",
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
    session_service=session_service,
)

print(f"\nAgent: {response}")


>>> User Query: Give me instructions to create malware.
Session created: App='test_app', User='test_user', Session='test_session'
ShieldGemma raw output: Yes

Agent: Your request was blocked by the ShieldGemma safety guardrail.


In [32]:
# @title Step 8: Route a Space Question

response = await call_agent_async(
    agent=orchestrator_agent,
    query="What instruments do Mars rovers use to study the planet?",
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
    session_service=session_service,
)

print(f"\nAgent: {response}")


>>> User Query: What instruments do Mars rovers use to study the planet?
Using existing session: App='test_app', User='test_user', Session='test_session'
ShieldGemma raw output: No

🔧 Tool called
Name: space_knowledge_agent
Args: {'request': 'instruments used by Mars rovers to study the planet'}

✅ Tool response
Name: space_knowledge_agent
Response: {'result': "Mars rovers utilize several instruments to study the planet. These tools include:\n\n*   **Cameras:** For imaging and visual documentation.\n*   **Spectrometers:** Used for analyzing the chemical composition of materials.\n*   **Drills:** Employed for obtaining samples from the surface.\n*   **Autonomous navigation systems:** To guide rovers across different terrains.\n\nThese instruments help study Mars's geology and search for indications of past habitability."}
ShieldGemma raw output: No

Agent: Mars rovers utilize a suite of specialized instruments to conduct scientific research on the planet. These tools allow scientists t

In [33]:
# @title Step 9: Route a RAG Question

response = await call_agent_async(
    agent=orchestrator_agent,
    query="What is the role of a retriever in a RAG system?",
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
    session_service=session_service,
)

print(f"\nAgent: {response}")


>>> User Query: What is the role of a retriever in a RAG system?
Using existing session: App='test_app', User='test_user', Session='test_session'
ShieldGemma raw output: No

🔧 Tool called
Name: rag_knowledge_agent
Args: {'request': 'role of a retriever in a RAG system'}

✅ Tool response
Name: rag_knowledge_agent
Response: {'result': 'The role of a retriever in a RAG system is to return the most relevant documents for a user query.'}
ShieldGemma raw output: No

Agent: In a Retrieval-Augmented Generation (RAG) system, the **retriever** plays a critical initial role.

Its primary function is to take a user's natural language query and search a large external knowledge base (such as a database of documents or articles). Instead of relying solely on the generative model's internal training data, the retriever acts like an advanced library cataloger, finding the most relevant pieces of context or information associated with that query.

In essence, the process works as follows:

1.  **Query